### Project - Airline Assistant using Database as Tool

#### Setting up the Database

In [2]:
import sqlite3

In [3]:
DB = 'prices.db'                        # sets the database filename

with sqlite3.connect(DB) as conn:       # opens the connection to the database with connection object conn
    cursor = conn.cursor()              # helper object to run the sql commands
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()                       # to save the changes permanently to the database

''' 
conn = the communication channel to the database (connection object)
cursor = the handheld tool you use to run commands through that channel (helper object)
'''

' \nconn = the communication channel to the database (connection object)\ncursor = the handheld tool you use to run commands through that channel (helper object)\n'

In [4]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?',(city.lower(), price, price))
        conn.commit()

In [5]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():       # to iterate over the dictionary {key:value} pairs
    set_ticket_price(city, price)

In [6]:
def get_ticket_price(city):
    print(f'DATABASE TOOL CALLED: Getting price for {city}', flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))      # if matching city found result will be price, otherwise None
        result = cursor.fetchone()                                                      # to return the first row of the result
        return f'Ticket price to {city} is ${result[0]}' if result else f'No price data available for {city}'
        
        
        # result might have 1-row with n-columns, so, the result is always is tuple
        # here we are accessing only the first column of the first row, that's why result[0]

In [7]:
get_ticket_price('delhi')

DATABASE TOOL CALLED: Getting price for delhi


'No price data available for delhi'

#### Setting up the LLM

In [8]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [9]:
load_dotenv(override=True)

GEMINI_BASE_URL = os.getenv('GEMINI_BASE_URL')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

gemini = OpenAI(base_url = GEMINI_BASE_URL, api_key = GEMINI_API_KEY)

if GEMINI_API_KEY:
    print(f'GEMINI API Key found and it starts with {GEMINI_API_KEY[0:3]}')
else:
    priint('GEMINI API KEY not found')

MODEL = 'gemini-3.5-flash-lite'

GEMINI API Key found and it starts with AQ.


In [10]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [11]:
# tool schema

my_tool_schema = {
    "name" : "get_ticket_price",
    "description" : "Get the price of a return ticket to the destination city",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "destination_city" : {
                "type" : "string", 
                "description" : "The city that the customer wants to travel to"
            },
        },
        "required" : ["destination_city"],
        "additionalProperties" : False
    }
}

In [12]:
tools = [{'type':'function', 'function':my_tool_schema}]

In [13]:
def chatbot(message, history):
    history = [{'role':h['role'], 'content':h['content']} for h in history]
    messages = [{'role':'system', 'content':system_message}] + history + [{'role':'user', 'content':message}]
    response = gemini.chat.completions.create(
        model = MODEL,
        messages = messages,
        tools = tools
    )

    while response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(
            model = MODEL,
            messages = messages,
            tools = tools
        )
    return response.choices[0].message.content


In [14]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == 'get_ticket_price':
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                'role' : 'tool',
                'content' : price_details,
                'tool_call_id' : tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chatbot).launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for tokyo
DATABASE TOOL CALLED: Getting price for sydney
DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Paris
